# TorchXRayVision DenseNet121 Backbone Verification

**Model**: `xrv.models.DenseNet(weights=...)` — multiple pre-training variants  
**Arch**: DenseNet121  
**Embed dim**: 1024 (output of `model.features()`, before classifier)  
**Input**: 1×224×224, range `[-1024, 1024]` (no RGB channels)  
**Install**: `pip install torchxrayvision`  
**Zero-shot**: No — vision-only

Available weight variants:

| Key | Training data |
|-----|---------------|
| `densenet121-res224-all` | NIH + CheXpert + MIMIC + PadChest (combined) |
| `densenet121-res224-rsna` | RSNA Pneumonia |
| `densenet121-res224-nih` | ChestX-ray14 |
| `densenet121-res224-pc` | PadChest |
| `densenet121-res224-chex` | CheXpert |
| `densenet121-res224-mimic_nb` | MIMIC-CXR (NegBio labels) |
| `densenet121-res224-mimic_ch` | MIMIC-CXR (CheXpert labels) |

Paper: [TorchXRayVision (Cohen et al., MIDL 2022)](https://arxiv.org/abs/2111.00595)

In [ ]:
%pip install -q torchxrayvision scikit-image torch

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

In [ ]:
import torch
import torchxrayvision as xrv

WEIGHTS = "densenet121-res224-all"   # change to explore other variants

model = xrv.models.DenseNet(weights=WEIGHTS)
model.eval()

print("pathologies:", model.pathologies)
print("num classes:", len(model.pathologies))

## 2. Image transform

TorchXRayVision uses a non-standard normalization:
- Input range **`[-1024, 1024]`** (mimics Hounsfield units)
- `xrv.datasets.normalize(img, maxval)` rescales a uint8/uint16 image to this range
- Single-channel (1, H, W) — **no RGB**

In [ ]:
import skimage
import numpy as np
from torchvision import transforms

def make_transform():
    return transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Load CXR, return [1, 1, 224, 224] batch tensor in [-1024, 1024]."""
    img = skimage.io.imread(path)
    img = xrv.datasets.normalize(img, maxval=255, reshape=True)  # → [-1024, 1024]
    if img.ndim == 3:        # RGB → grayscale mean
        img = img.mean(0, keepdims=True)
    img = transform(img)     # [1, 224, 224] numpy
    return torch.from_numpy(img).unsqueeze(0)   # [1, 1, 224, 224]

# Smoke-test with random data in correct range
dummy = torch.rand(1, 1, 224, 224) * 2048 - 1024   # [-1024, 1024]
print("dummy input:", dummy.shape, "range:", dummy.min().item(), dummy.max().item())

## 3. Extract embeddings via `model.features()`

In [ ]:
with torch.no_grad():
    feats  = model.features(dummy)    # [1, 1024]  ← dense feature vector
    logits = model(dummy)             # [1, num_pathologies]  ← classification logits

print("features shape:", feats.shape)   # [1, 1024]
print("logits shape:  ", logits.shape)  # [1, N_patho]
print("feature range: ", feats.min().item(), feats.max().item())

## Segmentation mode — spatial feature map (CNN)

TorchXRayVision uses **DenseNet121**, a CNN architecture — there are no patch tokens.
For dense prediction, capture the spatial feature map from the last dense block
before global average pooling via a forward hook on `model.features.norm5`.

For 224×224 input: `[B, 1024, 7, 7]` (stride 32 → 7×7 = 49 spatial positions).
This is analogous to ViT patch features, but coarser (stride 32 vs 16-px patches).

In [ ]:
import torch.nn.functional as F

# TorchXRayVision DenseNet applies global avg pool inside model.features().
# To get the spatial feature MAP before pooling, hook the last dense block's norm (norm5).
spatial_feat = {}

def _hook(module, inp, out):
    spatial_feat["map"] = out   # [B, C, H, W] — spatial feature map before pool

handle = model.features.norm5.register_forward_hook(_hook)
with torch.no_grad():
    _ = model(dummy)
handle.remove()

# DenseNet convention: apply ReLU to the raw norm5 output
feat_map = F.relu(spatial_feat["map"], inplace=False)

print(f"spatial feature map: {feat_map.shape}")
# [1, 1024, 7, 7] for 224px input (DenseNet stride=32: 224/32=7)
print("Note: this is a CNN feature map, not patch tokens — stride is 32.")

## 4. Multi-variant comparison — embedding consistency

In [ ]:
import torch.nn.functional as F

VARIANTS = [
    "densenet121-res224-all",
    "densenet121-res224-chex",
    "densenet121-res224-nih",
    "densenet121-res224-mimic_ch",
]

embeddings = {}
for w in VARIANTS:
    m = xrv.models.DenseNet(weights=w).eval()
    with torch.no_grad():
        f = m.features(dummy)           # [1, 1024]
        f = F.normalize(f, dim=-1)
    embeddings[w] = f
    print(f"{w}: shape={f.shape}")

# Cross-model cosine similarity (expect high overlap since same arch)
variants = list(VARIANTS)
print("\nCross-model cosine similarities on dummy image:")
for i in range(len(variants)):
    for j in range(i+1, len(variants)):
        sim = (embeddings[variants[i]] @ embeddings[variants[j]].T).item()
        print(f"  {variants[i].split('-')[-1]} vs {variants[j].split('-')[-1]}: {sim:.4f}")

## Notes for RadHarmony integration

```
model_call : None  (model itself is not called for embedding — use features())
            → wrap as: lambda m, x: m.features(x)
pool       : None
embed_dim  : 1024
transform  : normalize to [-1024,1024] → XRayCenterCrop → XRayResizer(224)
             output is single-channel (1, 224, 224) — NOT 3-channel
```

The single-channel input is the most important difference from other backbones — the MONAI transform must not repeat to 3 channels.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Hook `model.features.norm5` to get the pre-pool spatial feature map
- Shape: `[B, 1024, 7, 7]` for 224px input (CNN stride 32 → 7×7 spatial positions)
- Apply `F.relu(feat, inplace=False)` after the hook (DenseNet convention)
- ⚠️ CNN feature maps are stride-aligned, not patch-aligned — segmentation heads may need
  bilinear upsampling to match the mask resolution.

In [ ]:
# Replace with a real CXR path
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
print("input:", x.shape, "range:", x.min().item(), x.max().item())

model_all = xrv.models.DenseNet(weights="densenet121-res224-all").eval()
with torch.no_grad():
    feats = model_all.features(x)     # [1, 1024]
    preds = dict(zip(model_all.pathologies, model_all(x)[0].sigmoid().tolist()))

print("features shape:", feats.shape)
print("top predictions:",
      sorted(preds.items(), key=lambda kv: kv[1], reverse=True)[:5])

## Notes for RadHarmony integration

```
model_call : None  (model itself is not called for embedding — use features())
            → wrap as: lambda m, x: m.features(x)
pool       : None
embed_dim  : 1024
transform  : normalize to [-1024,1024] → XRayCenterCrop → XRayResizer(224)
             output is single-channel (1, 224, 224) — NOT 3-channel
```

The single-channel input is the most important difference from other backbones — the MONAI transform must not repeat to 3 channels.